# 03 · 학습 — **baseline** (`act` · `diffusion` · `smolvla` · `acm2`)

**프로토콜 (전 그룹 공통)** — 학습 **150k step · seed 4개 · lr 고정(sweep 없음)**,
eval **150k 체크포인트 × 5회 반복**(rep 마다 env seed 변경) → 모델당 4×5 = 20 run.

| 태그 | 비고 |
|---|---|
| `act` | Transformer 디코더 (원조) |
| `diffusion` | Diffusion Policy — **원 논문 기본값**(horizon 16 / n_action_steps 8 / **lr 1e-4**) |
| `smolvla` | VLA — `lerobot/smolvla_base` **파인튜닝** (**lr 1e-4**) |
| `acm2` | Mamba-2 디코더, carry off |
| (`act_te`) | 학습 X — eval 때 act 체크포인트에 TE 를 얹음 (`04_eval_baseline`) |

4모델 × 4 seed = **16잡**.

⚠️ **diffusion/smolvla 만 lr 1e-4** (나머지 1e-5). 남의 방법을 우리 lr 로 깎으면 baseline 이 불공정해져
리뷰어가 바로 지적함. 의도된 예외.

⚠️ **옛 런에서 baseline 을 가져다 쓰면 안 됨** — lr/step/seed 가 다르면 비교가 무효.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.MAIN_SIM        # 'insertion' (aloha)
SEEDS = cf.MAIN_SEEDS      # [0,1,2,3] — 은지와 분담하면 여기만 바꿈 (예: [0,1])
NGPU  = 8
TAGS = cf.GROUP_BASELINE   # ['act', 'diffusion', 'smolvla', 'acm2']

print('학습:', TAGS, '| 잡:', len(TAGS) * len(SEEDS))
for t in TAGS:
    pol, lr, K, extra, cp = cf.v23.MODEL_CONFIGS[t]
    print(f'  {t:<10} {pol:<12} lr={lr:<7} K={K:<4} {" ".join(extra)}')

## 커맨드 확인 (dry-run) — diffusion 은 chunk_size 대신 horizon, smolvla 는 --policy.path

In [ ]:
for t in TAGS:
    print(cf.make_train_cmd(t, seed=SEEDS[0], task=TASK, gpu_id=0))
    print()

## 학습

In [ ]:
jobs = cf.run_training(TAGS, SEEDS, task=TASK, ngpu=NGPU)

## 상태

In [ ]:
cf.print_training_status(jobs)
print()
cf.print_ckpt_status(TAGS, SEEDS, TASK)
print('\n다음: 04_eval_baseline')